# LangChain Memory 시리즈 6/7: ConversationSummaryMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- `ConversationSummaryMemory`가 요약을 누적해 나가는 방식을 설명할 수 있어요
- `ConversationSummaryBufferMemory`의 하이브리드 동작(버퍼 + 요약)을 이해할 수 있어요
- 두 클래스의 차이를 비교하고 상황에 맞게 선택할 수 있어요
- LLM 요약 비용과 토큰 절약 효과를 함께 고려할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| ConversationTokenBufferMemory | 토큰 수 기준으로 오래된 메시지를 삭제하는 메모리 (3번 노트북) |
| 텍스트 요약 (Summarization) | 긴 텍스트의 핵심 내용만 추려서 짧게 압축하는 기법 |
| max_token_limit | 버퍼를 유지할 최대 토큰 수. 초과 시 오래된 내용을 요약으로 압축 |

## 삭제 vs 요약, 무엇이 다를까요?

3번 노트북의 `ConversationTokenBufferMemory`는 토큰 한도 초과 시 오래된 메시지를 **완전히 삭제**해요. 초기 대화에서 나눈 중요한 맥락이 사라질 수 있어요.

`ConversationSummaryMemory`는 삭제 대신 **LLM이 요약**해서 보관해요. 대화 내용이 압축되어 손실은 있지만, 핵심 맥락은 유지돼요.

```mermaid
flowchart TD
    subgraph Token["TokenBufferMemory (3번)"]
        direction LR
        TK1[오래된 메시지] --"토큰 초과"--> TK2[완전 삭제]
    end

    subgraph Summary["SummaryMemory (이번 노트북)"]
        direction LR
        SK1[오래된 메시지] --"토큰 초과"--> SK2[LLM 요약]
        SK2 --> SK3[압축된 요약 보관]
    end

    subgraph Buffer["SummaryBufferMemory (하이브리드)"]
        direction LR
        BK1[오래된 메시지] --"토큰 초과"--> BK2[요약으로 압축]
        BK3[최근 메시지] --"토큰 이내"--> BK4[원문 그대로 유지]
    end

    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class TK1,SK1,BK1 error
    class TK2 error
    class SK2,SK3,BK2 process
    class BK3,BK4 output
```

> **참고**: 이 노트북은 레거시 메모리 클래스를 사용해요. LangChain 1.0.x에서는 요약 미들웨어 패턴이 권장돼요. 최신 방식은 13번 노트북을 참고하세요.

In [1]:
# ---------------------------------------------------
# 환경 변수 로드 및 요약 메모리 모듈 임포트
# ---------------------------------------------------
from dotenv import load_dotenv
# ⚠️ LangChain 1.0.x에서는 더 이상 `langchain.memory` 패키지를 사용하지 않습니다.
#    구식 메모리 클래스를 시연하기 위해 `langchain.memory`에서 불러옵니다.
from langchain.memory import ConversationSummaryMemory
# ChatOpenAI: 대화 요약 생성에 사용되는 LLM
from langchain_openai import ChatOpenAI

load_dotenv()

True

## 1. 기본 사용법 - 대화 요약 저장

`ConversationSummaryMemory`는 LLM을 사용하여 대화를 요약합니다.


### SummaryBufferMemory 구성 요소

```mermaid
graph TD
    LLM_SUM --> BUFFER_INIT["ConversationSummaryBufferMemory<br/>max_token_limit=200<br/>return_messages=True"]
    BUFFER_INIT --> STATES["최근 버퍼 + 요약 상태 동시 관리"]
```

In [3]:
# ============================================================
# TODO: ConversationSummaryMemory를 생성하세요
# 힌트: ChatOpenAI(temperature=0) → ConversationSummaryMemory(llm=llm, return_messages=True)
# 예상 결과: save_context() 호출마다 LLM이 기존 요약 + 새 대화를 재요약하여 history에 저장
# ============================================================

# 1단계: LLM 생성 (요약 생성에 사용)
# - temperature=0: 동일한 대화에 대해 일관된 요약 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: ConversationSummaryMemory 생성
# - llm 파라미터: 요약 생성에 사용하는 LLM (필수)
# - return_messages=True: SystemMessage 형태로 요약 반환 (체인 연결 시 편리)
memory = ConversationSummaryMemory(llm=llm)

### 💡 대화 저장 → 자동 요약

```mermaid
flowchart TD
    subgraph SaveLoop
        H[사용자 발화] --> SAVE["save_context()"]
        A[AI 응답] --> SAVE
        SAVE --> HIST[history 누적]
    end

    HIST --> LLM_SUMMARY[LLM으로 전체 대화 요약]
    LLM_SUMMARY --> SUMMARY[요약 텍스트]
```



### 💡 요약 조회 흐름

```mermaid
flowchart LR
    SUMMARY["저장된 요약(history)"] --> LOAD_SUM["load_memory_variables({})"]
    LOAD_SUM --> PRINT_SUM[요약 출력 + 특징 설명]
```

### 여러 대화 저장 및 요약 확인

여러 대화를 저장하면 자동으로 요약이 생성됩니다.


### 💡 SummaryBufferMemory 구성 요소

```mermaid
graph TD
    LLM_SUM --> BUFFER_INIT[ConversationSummaryBufferMemory(
        max_token_limit=200,
        return_messages=True
    )]
    BUFFER_INIT --> STATES[최근 버퍼 + 요약 상태 동시 관리]
```



In [5]:
# ---------------------------------------------------
# 여러 대화 저장 (자동 누적 요약)
# ---------------------------------------------------
# save_context() 호출마다 LLM이 기존 요약 + 새 대화를 합쳐 재요약
# 대화가 많아져도 history 토큰 수는 요약 길이로 제한됨
conversations = [
    {
        "human": "온라인 강의 플랫폼 구축에 대해 상담받고 싶어요.",
        "ai": "안녕하세요! 온라인 강의 플랫폼 구축을 도와드리겠습니다. 어떤 기능이 필요하신가요?"
    },
    {
        "human": "동영상 강의 재생, 수강생 관리, 과제 제출 기능이 필요해요.",
        "ai": "좋은 선택이에요! 동영상 재생은 HTML5 플레이어나 YouTube API를 사용할 수 있고, 수강생 관리는 회원가입/로그인 시스템과 학습 진도 추적 기능이 필요합니다."
    },
    {
        "human": "과제 제출 기능은 어떻게 구현하나요?",
        "ai": "과제 제출 기능은 파일 업로드, 제출 기한 설정, 제출 이력 관리 기능이 필요합니다. 또한 교수자가 평가하고 피드백을 줄 수 있는 기능도 포함하면 좋습니다."
    },
    {
        "human": "질문과 답변 게시판도 필요해요.",
        "ai": "네, Q&A 게시판은 질문 작성, 답변 작성, 검색 기능, 카테고리 분류 기능이 필요합니다. 실시간 알림 기능도 추가하면 사용자 경험이 향상됩니다."
    },
    {
        "human": "예산은 어느 정도 예상되나요?",
        "ai": "기본 기능만 구현한다면 약 3천만원 정도 예상됩니다. 추가 기능이나 고급 기능을 원하시면 더 비용이 발생할 수 있습니다."
    },
    {
        "human": "개발 기간은 얼마나 걸리나요?",
        "ai": "기본 기능만 구현한다면 약 3-4개월 정도 소요됩니다. 추가 기능이나 복잡한 기능이 많으면 더 오래 걸릴 수 있습니다."
    }
]

# 모든 대화 저장 (각 저장마다 누적 요약이 업데이트됨)
for conv in conversations:
    memory.save_context(
        inputs={"human": conv["human"]},
        outputs={"ai": conv["ai"]},
    )

### 💡 토큰 제한 미만일 때 데이터 흐름

```mermaid
flowchart TD
    INPUT_LOW[첫 대화 저장] --> SAVE_LOW["save_context()"]
    SAVE_LOW --> BUFFER[버퍼에 그대로 저장]
    BUFFER --> LOAD_LOW["load_memory_variables({})"]
    LOAD_LOW --> PRINT_LOW[history = 원문 그대로 출력]
```



In [6]:
# ---------------------------------------------------
# 누적 요약 결과 확인
# ---------------------------------------------------
# 6개 대화가 하나의 요약으로 압축되어 history에 저장됨
memory_vars = memory.load_memory_variables({})
memory_vars["history"]

'The human expresses a desire to consult about building an online lecture platform. The AI responds by offering assistance and inquires about the necessary features. The human specifies the need for video playback, student management, and assignment submission features. The AI agrees that these are good choices and suggests using HTML5 player or YouTube API for video playback, along with a registration/login system and progress tracking for student management. The human then asks how to implement the assignment submission feature, and the AI explains that it requires file upload, deadline setting, and submission history management, as well as a feature for instructors to evaluate and provide feedback. The human adds that a Q&A board is also needed, and the AI agrees, stating that it should include question and answer posting, search functionality, category classification, and suggests adding real-time notifications to enhance user experience. The human then inquires about the estimated

### 💡 토큰 제한 초과 시 하이브리드 동작

```mermaid
flowchart TD
    subgraph SaveMore
        EXTRA[추가 대화 리스트] --> LOOP[for conv in additional_conversations]
        LOOP --> SAVE_MORE[buffer_memory.save_context]
    end

    SAVE_MORE --> CHECK{"토큰 합계 > 200?"}
    CHECK -- 예 --> SUMMARIZE[LLM 요약으로 오래된 내용 압축]
    SUMMARIZE --> MERGE[요약 + 최근 버퍼 재구성]
    CHECK -- 아니오 --> BUFFER_KEEP[버퍼 유지]

    MERGE --> LOAD_HIGH["load_memory_variables({})"]
    BUFFER_KEEP --> LOAD_HIGH
    LOAD_HIGH --> PRINT_HIGH[history = 요약 + 최근 원문 출력]
```

## 2. ConversationSummaryBufferMemory

`ConversationSummaryBufferMemory`는 **최근 대화는 버퍼로 유지**하고, **오래된 대화는 요약**으로 저장하는 하이브리드 방식입니다.


In [9]:
# ============================================================
# TODO: ConversationSummaryBufferMemory를 max_token_limit=200으로 생성하세요
# 힌트: ConversationSummaryBufferMemory(llm=llm, max_token_limit=200, return_messages=True)
# 예상 결과: 토큰 200 이내이면 원문 유지, 초과 시 오래된 부분을 LLM이 요약으로 압축
# ============================================================

from langchain.memory import ConversationSummaryBufferMemory

# ConversationSummaryBufferMemory 생성
# - max_token_limit=200: 버퍼 임계값 (초과 시 오래된 대화를 요약으로 압축)
# - SummaryMemory와 달리 최근 대화는 원문 그대로 유지
buffer_memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=200, return_messages=True)


### 토큰 제한 미만일 때

토큰 제한을 넘지 않으면 대화가 그대로 저장됩니다.


In [11]:
# ---------------------------------------------------
# 토큰 제한 미만일 때: 원문 그대로 버퍼 유지
# ---------------------------------------------------
# 첫 대화 저장 후 200 토큰 미만이므로 요약 없이 원문이 그대로 history에 저장됨


buffer_memory.save_context(
    inputs={"human": "온라인 강의 플랫폼 구축에 대해 상담받고 싶어요."},
    outputs={"ai": "안녕하세요! 온라인 강의 플랫폼 구축을 도와드리겠습니다. 어떤 기능이 필요하신가요?"}
)

# 저장된 내용 확인 (아직 요약되지 않음)
memory_vars = buffer_memory.load_memory_variables({})

memory_vars

{'history': [HumanMessage(content='온라인 강의 플랫폼 구축에 대해 상담받고 싶어요.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='안녕하세요! 온라인 강의 플랫폼 구축을 도와드리겠습니다. 어떤 기능이 필요하신가요?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='온라인 강의 플랫폼 구축에 대해 상담받고 싶어요.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='안녕하세요! 온라인 강의 플랫폼 구축을 도와드리겠습니다. 어떤 기능이 필요하신가요?', additional_kwargs={}, response_metadata={})]}

### 토큰 제한 초과 시

토큰 제한을 초과하면 오래된 대화는 요약되고, 최근 대화는 버퍼로 유지됩니다.


In [13]:
# ---------------------------------------------------
# 토큰 제한 초과 시: 오래된 대화는 요약, 최근 대화는 버퍼 유지
# ---------------------------------------------------
# 추가 대화를 저장하면 200 토큰을 초과하게 되어 오래된 부분이 자동 요약됨
additional_conversations = [
    {
        "human": "동영상 강의 재생, 수강생 관리, 과제 제출 기능이 필요해요.",
        "ai": "좋은 선택이에요! 동영상 재생은 HTML5 플레이어나 YouTube API를 사용할 수 있고, 수강생 관리는 회원가입/로그인 시스템과 학습 진도 추적 기능이 필요합니다."
    },
    {
        "human": "과제 제출 기능은 어떻게 구현하나요?",
        "ai": "과제 제출 기능은 파일 업로드, 제출 기한 설정, 제출 이력 관리 기능이 필요합니다. 또한 교수자가 평가하고 피드백을 줄 수 있는 기능도 포함하면 좋습니다."
    },
    {
        "human": "질문과 답변 게시판도 필요해요.",
        "ai": "네, Q&A 게시판은 질문 작성, 답변 작성, 검색 기능, 카테고리 분류 기능이 필요합니다. 실시간 알림 기능도 추가하면 사용자 경험이 향상됩니다."
    },
    {
        "human": "예산은 어느 정도 예상되나요?",
        "ai": "기본 기능만 구현한다면 약 3천만원 정도 예상됩니다. 추가 기능이나 고급 기능을 원하시면 더 비용이 발생할 수 있습니다."
    }
]

# 모든 대화 저장 (200 토큰 초과 시 오래된 부분이 LLM 요약으로 압축됨)
for conv in conversations:
    buffer_memory.save_context(
        inputs={"human": conv["human"]},
        outputs={"ai": conv["ai"]}
    )

# 저장된 내용 확인 (아직 요약되지 않음)
memory_vars = buffer_memory.load_memory_variables({})

memory_vars

{'history': [SystemMessage(content='The human expresses a desire to seek advice about building an online course platform. The AI responds in Korean, offering to help with the construction of the online course platform and asks what features the human needs. The human specifies that they need features for video lecture playback, student management, and assignment submission. The AI suggests using an HTML5 player or YouTube API for video playback and mentions that student management requires a registration/login system and progress tracking features. The human then asks how to implement the assignment submission feature, and the AI explains that it requires file upload, deadline setting, submission history management, and a feature for instructors to evaluate and provide feedback. The human adds that they also need a Q&A forum, to which the AI states that the Q&A forum needs features for question and answer posting, search functionality, category classification, and suggests adding real-

## 3. 두 메모리 타입 비교

`ConversationSummaryMemory`와 `ConversationSummaryBufferMemory`의 차이를 비교해봅시다.


In [ ]:
print("=" * 60)
print("📊 메모리 타입별 비교")
print("=" * 60)

comparison_table = """
| 특징 | ConversationSummaryMemory | ConversationSummaryBufferMemory |
|------|-------------------------|-------------------------------|
| 저장 방식 | 모든 대화를 요약 | 최근 대화는 버퍼, 오래된 대화는 요약 |
| 토큰 관리 | 요약만 저장하여 토큰 절약 | 버퍼 + 요약으로 균형 유지 |
| 최근 정보 | 요약에 포함 (세부 정보 손실 가능) | 버퍼에 보존 (세부 정보 유지) |
| 오래된 정보 | 요약으로 압축 | 요약으로 압축 |
| 적합한 경우 | 매우 긴 대화, 전체 요약 필요 | 긴 대화, 최근 정보 중요 |
"""

print(comparison_table)


## 핵심 정리

```python
# ConversationSummaryMemory 기본 사용법
from langchain.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 대화 저장 (자동 요약)
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 요약 확인
summary = memory.load_memory_variables({})["history"]
```

```python
# ConversationSummaryBufferMemory 기본 사용법
from langchain.memory import ConversationSummaryBufferMemory

buffer_memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=200,  # 토큰 제한
    return_messages=True
)
```

### 주요 특징
- ✅ **대화 요약**: LLM을 사용하여 대화를 요약하여 저장
- ✅ **토큰 절약**: 전체 대화 대신 요약만 저장하여 토큰 사용량 감소
- ✅ **맥락 보존**: 요약을 통해 핵심 정보는 유지
- ✅ **긴 대화 적합**: 매우 긴 대화에서 유용

### ConversationSummaryBufferMemory의 장점
- ✅ **하이브리드 방식**: 최근 대화는 버퍼로, 오래된 대화는 요약으로 저장
- ✅ **세부 정보 보존**: 최근 대화의 세부 정보를 유지하면서 토큰 절약
- ✅ **유연한 관리**: `max_token_limit`으로 버퍼 크기 조절 가능

### 주의사항
- ⚠️ **LLM 필요**: 요약 생성에 LLM 인스턴스가 필요함
- ⚠️ **비용**: 요약 생성에 LLM 호출이 필요하므로 비용 발생
- ⚠️ **정보 손실**: 요약 과정에서 일부 세부 정보가 손실될 수 있음
- ⚠️ **요약 품질**: LLM의 성능에 따라 요약 품질이 달라짐